# guide

In this notebook we experiment with the guidance of the det-gen ArchesWeather model.

In [2]:
%load_ext autoreload

%autoreload 2

In [3]:
cd ..

/Users/alain/Desktop/master-thesis/guiding-diffusion-based-weather-models


In [4]:
from pathlib import Path 
from typing import Callable

import pandas as pd
import xarray as xr
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from geoarches.lightning_modules import load_module
from geoarches.dataloaders.era5 import Era5Forecast

from diffusers.schedulers import FlowMatchEulerDiscreteScheduler

In [5]:
##### setup

ds = Era5Forecast(
    path="data/era5_240/full",  # default path
    domain="test", # domain to consider. domain = 'test' loads the 2020 period
    load_prev=True,  # whether to load previous state
    norm_scheme="pangu",  # default normalization scheme
    lead_time_hours=6
)

device = "mps"

gen_model, gen_config = load_module("archesweathergen")
gen_model = gen_model.to(device)

sample_state = ds[0]

4it [00:00, 11.43it/s]


start time 2019-12-31T18:00:00


/Users/alain/Desktop/master-thesis/guiding-diffusion-based-weather-models/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Restored from modelstore/archesweather-m-seed0/checkpoints/checkpoint.ckpt
Restored from modelstore/archesweather-m-seed1/checkpoints/checkpoint.ckpt
Restored from modelstore/archesweather-m-skip-seed0/checkpoints/checkpoint.ckpt
Restored from modelstore/archesweather-m-skip-seed1/checkpoints/checkpoint.ckpt
Restored from modelstore/archesweathergen/checkpoints/checkpoint.ckpt


In [6]:
type(gen_model.backbone)

geoarches.backbones.archesweather.ArchesWeatherCondBackbone

In [7]:

# batch = {k: v[None].to(device) for k, v in sample_state.items()}

# scheduler = FlowMatchEulerDiscreteScheduler(
#             num_train_timesteps=1000
#         )
# num_steps = 25
# scheduler.set_timesteps(num_steps)

# # conditional by default
# times = pd.to_datetime(batch["timestamp"].cpu().numpy() * 10**9).tz_localize(None)
# month = torch.tensor(times.month).to(device)
# month_emb = gen_model.month_embedder(month)
# hour = torch.tensor(times.hour).to(device)
# hour_emb = gen_model.hour_embedder(hour)
# timestep_emb = gen_model.timestep_embedder(timesteps)

# cond_emb = month_emb + hour_emb + timestep_emb

# x = gen_model.embedder.encode(batch["state"], input_state)

# gen_model.backbone(sample_state)

In [8]:
# optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
# # TODO: how do we initialize the optimizer since we are updating X?
# #       maybe I can write the optimization loop myself


## Generation guidance at time $t$

Single pass through the backbone computing the gradient with respect to the noisy input.

In [9]:
# device = "mps"

# def average_over_mask(mask, state):
#     return torch.mean(mask * state)

# def guidance_term(drift, avg_value):
#     return avg_value + drift * torch.abs(avg_value)

# # fixed reference state
# current_state = torch.tensor(
#     [[1.0, 2.0],
#      [3.0, 4.0]],
#     device=device,
#     dtype=torch.float32,
# )

# # noisy state we want gradients for
# noisy_state = torch.tensor(
#     [[1.2, 1.8],
#      [2.7, 4.3]],
#     device=device,
#     dtype=torch.float32,
#     requires_grad=True,
# )

# # fixed mask
# mask = torch.tensor(
#     [[1.0, 0.0],
#      [1.0, 0.0]],
#     device=device,
#     dtype=torch.float32,
# )

# # single drift value
# drift = torch.tensor(0.5, device=device, dtype=torch.float32)

# # target computed from the fixed current state
# current_init_avg = average_over_mask(mask, current_state)
# guidance_term_ = guidance_term(drift, current_init_avg)

# # aggregate computed from the noisy state
# gen_agg_term = average_over_mask(mask, noisy_state)

# # loss compares noisy aggregate to fixed target
# loss = torch.nn.functional.mse_loss(gen_agg_term, guidance_term_)

# print("current_init_avg:", current_init_avg)
# print("target:", guidance_term_)
# print("gen_agg_term:", gen_agg_term)
# print("loss:", loss)

# loss.backward()

# print("gradient wrt noisy_state:")
# print(noisy_state.grad)

## M step SGD 
... to compute the full guidance state, which is not a single scalar!

The difference here is that we are guiding the generation towards the state that generates what we want. In contrast, if the model would be fully generative, we would guide using a single scalar.

In [10]:
##### sample

seed = 0
num_steps = 25
scale_input_noise = 1.05

batch = {k: v[None].to(device) for k, v in sample_state.items()}

sample = gen_model.guided_sampling(
    batch, seed=seed, num_steps=num_steps, scale_input_noise=scale_input_noise, 
    guidance_term=torch.tensor(0.5, device=device, dtype=torch.float32), 
    mask=None, 
).cpu()

loop_batch {'timestamp': tensor([1577858400], device='mps:0', dtype=torch.int32), 'state': TensorDict(
    fields={
        level: Tensor(shape=torch.Size([1, 6, 13, 121, 240]), device=mps:0, dtype=torch.float32, is_shared=False),
        surface: Tensor(shape=torch.Size([1, 4, 1, 121, 240]), device=mps:0, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([1]),
    device=mps,
    is_shared=False), 'lead_time_hours': tensor([6], device='mps:0', dtype=torch.int32), 'prev_state': TensorDict(
    fields={
        level: Tensor(shape=torch.Size([1, 6, 13, 121, 240]), device=mps:0, dtype=torch.float32, is_shared=False),
        surface: Tensor(shape=torch.Size([1, 4, 1, 121, 240]), device=mps:0, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([1]),
    device=mps,
    is_shared=False), 'pred_state': TensorDict(
    fields={
        level: Tensor(shape=torch.Size([1, 6, 13, 121, 240]), device=mps:0, dtype=torch.float32, is_shared=False),
        surface: Ten

  0%|          | 0/25 [00:07<?, ?it/s]


KeyboardInterrupt: 

```bash
M = 10

def updated_states(input_state, pred_state):
    input_state["prev_state"] = input_state["state"]
    input_state["state"] = pred_state
    return input_state

def gen_guidance_state(input_state):
    X_hat = det(input_state)
    input_state = updated_states(input_state, pred_state)
    X_hat = det(input_state)
    X_tilde = SGD(X_hat, input_state)
    return X_tilde

def SGD(X_hat):
    for m in range(M):
        X_input["prev_state"] = X_input["prev_state"] - eta * grad{X_input["prev_state"]}(X_hat)
    # NOTE: we will generate using this state for guidance
    return X_input["prev_state"]

X_input = start_state()
for n in N:
    if n > 1
        X_input = updated_states(X_input, X_bar)

    X_tilde = gen_guidance_state(X_input)
    # NOTE: we still use the unchanged X_input for generation but guide towards X_tilde
    X_bar = gen_guide(X_input, X_tilde)
    save_states(X_tilde, X_bar)
```